# LegalMitra: Zero-Error LoRA Adapter Merge & 4-bit GGUF Export

This notebook merges the fine-tuned LegalMitra LoRA adapter into the base `unsloth/Meta-Llama-3.1-8B-Instruct` model and exports a quantized **4-bit GGUF (`legalmitra-8b-q4_k_m.gguf`, ~4.6 GB)** for offline, CPU-only local execution.

### Bulletproof Engineering Safeguards:
1. **Zero-Restart Clean Imports:** All merge operations run in an isolated Python subprocess with `USE_TF=0`, avoiding Kaggle's known in-kernel `GenerationMixin` import bug.
2. **Strict Disk Quota Management:** Kaggle free instances provide ~19.5 GB disk space. To prevent `OSError: Not enough free space` during the 16 GB FP16 write, the HuggingFace cache is purged immediately after model loading, and shards are deleted immediately before quantization.
3. **Native CMake llama.cpp Quantizer:** Compiles CPU binaries directly without missing CUDA driver linker errors.

**Hardware:** Kaggle GPU (T4 x2 or P100) | **Internet:** On

## Step 1: Install Dependencies, Build llama.cpp & Prepare Environment

In [ ]:
# 1. Clean up incompatible dependencies
!pip uninstall -y torchao
!pip install -q "transformers>=4.44.0" "peft>=0.12.0" "accelerate>=0.33.0" sentencepiece huggingface_hub gguf

# 2. Clone llama.cpp and compile quantizer & cli with CMake
!rm -rf /kaggle/working/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp -DGGML_CUDA=OFF
!cmake --build /kaggle/working/llama.cpp/build --config Release -j4 --target llama-quantize llama-cli
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

print("SUCCESS: Step 1 completed. Environment configured and llama.cpp built cleanly.")

## Step 2: Locate Uploaded LoRA Adapter Dataset

In [ ]:
import os
import glob

# Find adapter across attached Kaggle datasets or local working dir
adapter_candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

if adapter_candidates:
    adapter_dir = os.path.dirname(adapter_candidates[0])
    print(f"Found LegalMitra LoRA adapter at: {adapter_dir}")
else:
    adapter_dir = '/kaggle/working/legalmitra_llama3_lora_final'
    if not os.path.exists(adapter_dir):
        raise FileNotFoundError(
            "LoRA adapter not found! Please attach your 'legalmitra-lora-adapter' dataset "
            "via '+ Add Input' on the right sidebar."
        )
    print(f"Using local session adapter at: {adapter_dir}")

print("Adapter files:", os.listdir(adapter_dir))

with open('/kaggle/working/adapter_path.txt', 'w') as f:
    f.write(adapter_dir)
print("SUCCESS: Step 2 completed. Adapter path registered.")

## Step 3: Isolated LoRA Merge & Safe Shard Export

In [ ]:
# Write isolated merge runner script to avoid Jupyter kernel namespace corruption
merge_script = '''
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Safeguard for torchao incompatibility in Kaggle's environment
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

with open("/kaggle/working/adapter_path.txt", "r") as f:
    adapter_dir = f.read().strip()

base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"
merged_output_dir = "/kaggle/working/legalmitra_merged_shards"

print(f"[1/4] Loading base model ({base_model_name}) into GPU...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"[2/4] Applying LegalMitra LoRA adapter from {adapter_dir}...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("[3/4] Merging LoRA weights directly into base weights...")
merged_model = model.merge_and_unload()

print(f"[4/4] Saving merged FP16 model to {merged_output_dir} in compact shards...")
merged_model.save_pretrained(merged_output_dir, max_shard_size="1800MB", safe_serialization=True)
tokenizer.save_pretrained(merged_output_dir)
print("SUCCESS: Model merged and saved successfully.")
'''

with open('/kaggle/working/merge_runner.py', 'w') as f:
    f.write(merge_script)

# Execute merge in isolated process
!python3 /kaggle/working/merge_runner.py

# Immediately purge HF download cache (~15 GB) to free disk space for GGUF quantization
!rm -rf /root/.cache/huggingface
print("SUCCESS: Step 3 completed. Merged shards saved and cache purged.")

## Step 4: Convert Merged Shards to GGUF & Quantize to 4-bit (q4_k_m)

In [ ]:
import os

fp16_gguf_path = "/kaggle/working/temp_fp16.gguf"
q4_gguf_path = "/kaggle/working/legalmitra-8b-q4_k_m.gguf"
quantize_bin = "/kaggle/working/llama.cpp/build/bin/llama-quantize"

print("[1/3] Converting HF shards to intermediate FP16 GGUF...")
!python3 /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/legalmitra_merged_shards \
    --outfile {fp16_gguf_path} \
    --outtype f16

# Delete raw HF shards immediately so total disk usage stays well below the 19.5 GB quota!
print("[2/3] Freeing raw shards from disk before quantization...")
!rm -rf /kaggle/working/legalmitra_merged_shards

print("[3/3] Quantizing FP16 GGUF to 4-bit Medium (q4_k_m)...")
!{quantize_bin} {fp16_gguf_path} {q4_gguf_path} q4_k_m

# Delete intermediate FP16 GGUF file and cleanup
if os.path.exists(fp16_gguf_path):
    os.remove(fp16_gguf_path)
if os.path.exists('/kaggle/working/merge_runner.py'):
    os.remove('/kaggle/working/merge_runner.py')

if os.path.exists(q4_gguf_path):
    q4_size_gb = os.path.getsize(q4_gguf_path) / (1024 ** 3)
    print("\n" + "="*65)
    print(f"SUCCESS: Standalone model ready: {q4_gguf_path}")
    print(f"Final 4-bit Quantized Size: {q4_size_gb:.2f} GB")
    print("="*65)
else:
    raise RuntimeError("Quantization failed to produce legalmitra-8b-q4_k_m.gguf!")

## Step 5: Sanity Test Inference & Download Instructions

In [ ]:
test_prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat happens in case of minor theft under the Bharatiya Nyaya Sanhita, 2023?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

cli_bin = "/kaggle/working/llama.cpp/build/bin/llama-cli"

print("Running test legal inference on newly created GGUF model:\n")
!{cli_bin} \
    -m /kaggle/working/legalmitra-8b-q4_k_m.gguf \
    -p "{test_prompt}" \
    -n 128 \
    --temp 0.2

print("\n" + "="*65)
print("ALL DONE! In the Kaggle right panel under 'Output', download:")
print("  -> legalmitra-8b-q4_k_m.gguf (~4.6 GB)")
print("Save it into your local LegalMitra/models/ directory.")
print("="*65)